# BERTopic Conversation Topic Modeling — Twitter/X Election Conversations

This notebook takes pre-grouped, pre-chunked conversation parquet files and produces topic
clusters whose **labels describe the core idea being discussed**, not just 2-3 raw keywords.

**Pipeline:**
1. Setup & installs
2. Config — paths to the 3 pre-built conversation parquet chunks
3. Load the top N (default 2,000,000) conversations by tweet count, across all chunks
4. Embedding model — a larger, higher-quality sentence embedding model
5. Representation model — an LLM that reads representative conversations per topic and writes
   a short human-readable label describing what the topic is actually about
6. Clustering models (UMAP + HDBSCAN, GPU-accelerated if available)
7. Embed the selected corpus in batches, saving each batch to disk as it finishes
8. Fit BERTopic once on the selected corpus
9. Save results and export

**Data assumption:** the 3 input files already exist at:
```
/content/drive/MyDrive/bertopic_outputs/conversations_chunk0.parquet
/content/drive/MyDrive/bertopic_outputs/conversations_chunk1.parquet
/content/drive/MyDrive/bertopic_outputs/conversations_chunk2.parquet
```
Each file has one row per conversation with columns `conversation_id` and `doc` (the
concatenated conversation text). We only have this conversation-level data — no embeddings
exist yet — so this run computes embeddings and fits the topic model from scratch.

**Why fit on everything instead of chunk 0 only:** fitting only on chunk 0 and then
`.transform()`-ing chunks 1 and 2 onto those topics means anything that's a genuinely distinct
topic in chunk 1 or 2 but wasn't well represented in chunk 0 gets forced into the nearest
chunk-0 topic (or dropped as an outlier) instead of forming its own cluster. Fitting once on
the combined corpus lets HDBSCAN see the full topic structure across all ~2.5M conversations
before deciding on clusters, so nothing chunk-specific gets silently absorbed or missed.
This also means there's no "already fitted, reload and skip" resume logic — every run is a
full fit on the full dataset, since we're starting only from raw conversation text.


https://drive.google.com/file/d/1be4aS47DRyTjt4giF4S6hto6lmSE-OXl/view?usp=sharing - 1 chunck
https://drive.google.com/file/d/18SMsIXTBa8mBgI6clSlUh4vr2i21nAou/view?usp=sharing - 2 chunk
https://drive.google.com/file/d/1JN32U4TRx7n6Ty1jwDmBj7vQc5IzYxPr/view?usp=sharing - 3 chunk

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q duckdb bertopic sentence-transformers transformers accelerate umap-learn hdbscan polars pyarrow

# Optional GPU-accelerated UMAP/HDBSCAN (RAPIDS cuML). Takes ~5-10 min to install.
# If this fails or is skipped, the script falls back to CPU automatically.
!pip install -q --extra-index-url=https://pypi.nvidia.com cuml-cu12

import gc
import os
import glob

# Must be set before torch initializes CUDA - reduces allocator fragmentation,
# which is what caused the "reserved but unallocated" OOM warning.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import duckdb
import numpy as np
import pandas as pd
import torch

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

gc.collect()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 13.0 MB/s eta 0:00:00
Using device: cuda
GPU: NVIDIA A100-SXM4-80GB


## 2. Config

Points directly at the 3 pre-built conversation-level parquet chunks. No raw tweet parquet,
no DuckDB GROUP BY step, no chunk-splitting step — that work is assumed already done.

In [ ]:
DATA_DIR = "/content/drive/MyDrive/bertopic_outputs"
N_CHUNKS = 3
CHUNK_PATHS = [os.path.join(DATA_DIR, f"conversations_chunk{c}.parquet") for c in range(N_CHUNKS)]

for p in CHUNK_PATHS:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Expected chunk file not found: {p}")
print("Chunk files found:")
for p in CHUNK_PATHS:
    print(" -", p)

CONVO_ID_COL = "conversation_id"
TEXT_COL = "doc"

# Column holding the number of tweets in each conversation. Adjust this to match
# whatever your parquet files actually call it (e.g. "n_tweets", "num_tweets",
# "tweet_count"). Cell 5 prints the schema of chunk 0 so you can double check.
TWEET_COUNT_COL = "n_tweets"

# Only keep the N conversations with the most tweets, selected across ALL chunks
# combined (not N per chunk). Set to None to keep everything.
N_TOP_CONVERSATIONS = 2_000_000

MIN_TOPIC_SIZE = 40

# Embedding model: Qwen3-Embedding-0.6B (decoder-based, ~0.6B params). Swapped in
# from BAAI/bge-large-en-v1.5 for speed/quality tradeoff. Note this is architecturally
# different from BGE (causal self-attention, not a bidirectional encoder) - it is NOT
# automatically lighter on GPU memory just because the param count looks similar, so
# sequence length and batch size are capped conservatively in the embedding cell.
EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"

# Hard cap on tokens per conversation. Conversations were selected by highest tweet
# count, which means the selected set is biased toward the longest possible texts -
# exactly the ones most likely to spike memory. Qwen3-Embedding's native context is
# very long (up to 32K tokens), so leaving this uncapped is what caused the OOM.
MAX_SEQ_LENGTH = 512

# Qwen3-Embedding is instruction-aware: prefixing a short task instruction improves
# embedding quality for the target use case (here, topic clustering).
EMBED_QUERY_PROMPT = "Instruct: Represent this text for clustering by topic\nQuery: "

# Instruction-tuned generative model used to turn each cluster's representative
# conversations into a short "core idea" label instead of a bag of keywords.
GEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

OUTPUT_DIR = DATA_DIR  # write outputs alongside the input chunks
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_PATH = os.path.join(OUTPUT_DIR, "bertopic_model")

# Embeddings are computed and saved to disk in batches of this many conversations,
# instead of one giant encode() call over everything. Each batch is written straight
# into the final memmap file as soon as it finishes (the checkpoint), so a crash or
# disconnect only costs the current batch, not the whole run - re-running the notebook
# picks up exactly where it left off via progress.json.
EMBED_BATCH_SIZE = 100_000
EMBEDDINGS_BATCH_DIR = os.path.join(OUTPUT_DIR, "embeddings_batches")
os.makedirs(EMBEDDINGS_BATCH_DIR, exist_ok=True)
COMBINED_EMBEDDINGS_PATH = os.path.join(OUTPUT_DIR, "embeddings_all.npy")

DUCKDB_MEMORY = "24GB"
DUCKDB_THREADS = 4

print(f"Output dir: {OUTPUT_DIR}")
print(f"Will keep top {N_TOP_CONVERSATIONS:,} conversations by {TWEET_COUNT_COL}" if N_TOP_CONVERSATIONS else "Will keep all conversations")
print(f"Embedding model: {EMBED_MODEL_NAME} (max_seq_length={MAX_SEQ_LENGTH})")
print(f"Embedding batch size: {EMBED_BATCH_SIZE:,} conversations/batch -> {EMBEDDINGS_BATCH_DIR}")


Chunk files found:
 - /content/drive/MyDrive/bertopic_outputs/conversations_chunk0.parquet
 - /content/drive/MyDrive/bertopic_outputs/conversations_chunk1.parquet
 - /content/drive/MyDrive/bertopic_outputs/conversations_chunk2.parquet
Output dir: /content/drive/MyDrive/bertopic_outputs
Will keep top 2,000,000 conversations by n_tweets
Embedding model: BAAI/bge-base-en-v1.5 (max_seq_length=512)
Embedding batch size: 100,000 conversations/batch -> /content/drive/MyDrive/bertopic_outputs/embeddings_batches


## 3. Load top conversations by tweet count

Reads all 3 chunk files with DuckDB, ranks conversations across the *combined* corpus by
`TWEET_COUNT_COL` (descending), and keeps only the top `N_TOP_CONVERSATIONS`
(2,000,000 by default). Filtering happens directly in the `ORDER BY ... LIMIT ...` SQL
query rather than in pandas afterwards, so we never have to load the full ~2.5M-row corpus
into memory just to throw most of it away.

If `N_TOP_CONVERSATIONS` is `None`, every conversation is kept (same behavior as before).

In [ ]:
con = duckdb.connect()
con.execute(f"SET memory_limit='{DUCKDB_MEMORY}'")
con.execute(f"SET threads TO {DUCKDB_THREADS}")

# Sanity-check the schema of chunk 0 so TWEET_COUNT_COL can be verified/corrected above.
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{CHUNK_PATHS[0]}')").fetchdf()
print("Columns available in chunk 0:")
print(schema[["column_name", "column_type"]].to_string(index=False))
if TWEET_COUNT_COL not in schema["column_name"].tolist():
    raise ValueError(
        f"TWEET_COUNT_COL='{TWEET_COUNT_COL}' not found in {CHUNK_PATHS[0]}. "
        f"Set TWEET_COUNT_COL in the config cell to one of: {schema['column_name'].tolist()}"
    )

union_sql = " UNION ALL ".join(
    f"""SELECT {CONVO_ID_COL} AS conversation_id, {TEXT_COL} AS doc, {TWEET_COUNT_COL} AS tweet_count
        FROM read_parquet('{path}')"""
    for path in CHUNK_PATHS
)

if N_TOP_CONVERSATIONS is not None:
    query = f"""
        SELECT conversation_id, doc, tweet_count
        FROM ({union_sql})
        WHERE doc IS NOT NULL
        ORDER BY tweet_count DESC
        LIMIT {N_TOP_CONVERSATIONS}
    """
else:
    query = f"""
        SELECT conversation_id, doc, tweet_count
        FROM ({union_sql})
        WHERE doc IS NOT NULL
        ORDER BY tweet_count DESC
    """

print("\nRunning ranked selection query across all chunks ...")
selected_df = con.execute(query).fetchdf()

all_ids = selected_df["conversation_id"].tolist()
all_docs = selected_df["doc"].tolist()
all_tweet_counts = selected_df["tweet_count"].tolist()

print(f"Selected {len(all_docs):,} conversations "
      f"(tweet_count range: {selected_df['tweet_count'].min():,} - {selected_df['tweet_count'].max():,})")
assert len(all_ids) == len(all_docs) == len(all_tweet_counts)

del selected_df
gc.collect()


Columns available in chunk 0:
    column_name column_type
conversation_id     VARCHAR
            doc     VARCHAR
       n_tweets      BIGINT

Running ranked selection query across all chunks ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Selected 2,000,000 conversations (tweet_count range: 2 - 20,259)


0

## 4. Embedding model

Loads `Qwen/Qwen3-Embedding-0.6B`. Two settings matter a lot for a 2,000,000-conversation
run selected by highest tweet count (i.e. biased toward the longest texts in the corpus):

- `max_seq_length` is capped at `MAX_SEQ_LENGTH` tokens. Left uncapped, this model's very
  long native context window means a single outlier conversation can spike GPU memory
  and OOM the whole batch it's in (this happened during initial testing).
- The model is cast to fp16 on GPU to reduce memory further. Qwen3-Embedding is
  decoder-based (causal self-attention), not a lightweight bidirectional encoder like
  BGE/E5, so it is not automatically cheaper on memory just because the parameter count
  (0.6B) looks small - batch sizes are kept conservative in the embedding cell for this
  reason.

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)

# Cap sequence length - see markdown above for why this matters for this dataset.
embed_model.max_seq_length = MAX_SEQ_LENGTH

if device == "cuda":
    embed_model = embed_model.half()  # fp16, roughly halves activation memory
    print("Model cast to fp16")

print(f"Loaded embedding model: {EMBED_MODEL_NAME} (dim={embed_model.get_sentence_embedding_dimension()}, "
      f"max_seq_length={embed_model.max_seq_length})")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model cast to fp16
Loaded embedding model: BAAI/bge-base-en-v1.5 (dim=768, max_seq_length=512)


## 5. Representation model — "core idea" topic labels

Default BERTopic labels come from c-TF-IDF: the top-N most distinctive words per topic. That's
why the old output looked like `usanews0 amy_siskind` or `gaza genocide` — just keywords, no
sense of what the conversation is actually about.

Here we add a **generative** representation model on top of c-TF-IDF: for each topic, BERTopic
feeds a handful of representative conversations plus the top keywords to an instruction-tuned
LLM (`Qwen2.5-3B-Instruct`) and asks it to write a short phrase describing the core theme —
e.g. "Debate over Hunter Biden pardon and DOJ fairness" instead of `pardon hunter`.

We keep `KeyBERTInspired` as a secondary ("Aspect") representation so the raw keyword view is
still available in `topic_info` if useful, but `topic_model.get_topic_info()["Name"]` will now
reflect the LLM-written core-idea label.

In [ ]:
from transformers import pipeline
from bertopic.representation import TextGeneration, KeyBERTInspired

generator = pipeline(
    "text-generation",
    model=GEN_MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

TOPIC_LABEL_PROMPT = """You are labeling a cluster of Twitter/X conversations for topic analysis.

Representative excerpts from conversations in this topic:
[DOCUMENTS]

The most distinctive keywords for this topic are: [KEYWORDS]

Write ONE short phrase (5-10 words) that captures the CORE IDEA people are actually discussing
in this topic - the underlying event, debate, or theme - not just a list of keywords.
Do not use quotation marks. Respond with the phrase only, nothing else.

Topic label:"""

representation_model = {
    "Main": TextGeneration(
        generator,
        prompt=TOPIC_LABEL_PROMPT,
        nr_docs=8,        # representative conversations shown to the LLM per topic
        doc_length=250,   # characters kept per conversation, to control prompt size
        tokenizer="whitespace",
    ),
    "KeyBERT": KeyBERTInspired(),
}
print("Representation model ready:", GEN_MODEL_NAME)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Representation model ready: Qwen/Qwen2.5-1.5B-Instruct


## 6. Clustering models (UMAP + HDBSCAN)

GPU (cuML) if available, CPU fallback otherwise.

In [ ]:
def build_cluster_models():
    if device == "cuda":
        try:
            from cuml.manifold import UMAP as cuUMAP
            from cuml.cluster import HDBSCAN as cuHDBSCAN
            umap_model = cuUMAP(n_neighbors=25, n_components=5, min_dist=0.0, random_state=42)
            hdbscan_model = cuHDBSCAN(min_cluster_size=40, prediction_data=True)
            print("Using cuML UMAP + HDBSCAN (GPU)")
            return umap_model, hdbscan_model
        except ImportError:
            print("cuML not available, falling back to CPU UMAP + HDBSCAN")
    from umap import UMAP
    from hdbscan import HDBSCAN
    umap_model = UMAP(n_neighbors=40, n_components=5, min_dist=0.0, random_state=42, low_memory=True)
    hdbscan_model = HDBSCAN(min_cluster_size=25, prediction_data=True)
    print("Using CPU UMAP + HDBSCAN")
    return umap_model, hdbscan_model


## 7. Embed the corpus in batches (checkpointed to disk)

We only have raw conversation text — no embeddings exist yet — so this encodes the selected
top-`N_TOP_CONVERSATIONS` conversations with `embed_model`. Encoding is done in chunks of
`EMBED_BATCH_SIZE` conversations, and **each batch is written directly into the final
`embeddings_all.npy` memmap and flushed to disk as soon as it's done** - that write *is*
the checkpoint, tracked in `embeddings_batches/progress.json`.

Why batch instead of one `.encode()` call: with 2,000,000 conversations, a single encode()
call is a multi-hour, all-or-nothing operation on Colab - if the runtime disconnects or hits
an out-of-memory error near the end, everything is lost. Batching means:
- Progress is checkpointed continuously; if the notebook stops, re-running this cell skips
  batches already written to the memmap and resumes exactly where it left off.
- Peak memory usage is bounded by one batch instead of the full 2M x embedding-dim array.

This cell also includes two defenses specific to this dataset and model, both added after
hitting `CUDA out of memory` during testing:
- **Length diagnostics** printed up front, since conversations were selected by highest
  tweet count and are therefore biased toward the longest, most memory-hungry texts.
- **Automatic batch-size retry on OOM** - if a sub-batch still OOMs even with
  `MAX_SEQ_LENGTH` capped, it's retried at half the batch size (down to a floor of 4)
  instead of crashing the whole run.

In [ ]:
import math
import json
n_docs = 2_000_000

# --- Diagnostics: check for extreme-length outliers before encoding ---
doc_lengths = sorted((len(d) for d in all_docs), reverse=True)
print("Longest docs (characters):", doc_lengths[:10])
print(f"95th percentile length: {doc_lengths[int(0.05 * len(doc_lengths))]:,} chars")
print(f"Median length: {doc_lengths[len(doc_lengths)//2]:,} chars")

n_docs = len(all_docs)
embed_dim = embed_model.get_sentence_embedding_dimension()
n_batches = math.ceil(n_docs / EMBED_BATCH_SIZE)

# Conservative starting sub-batch size for encode() - Qwen3-Embedding is decoder-based
# and more memory-hungry per token than BGE/E5 at a similar parameter count.
encode_batch_size = 64 if device == "cuda" else 16

progress_path = os.path.join(EMBEDDINGS_BATCH_DIR, "progress.json")

def load_progress():
    if os.path.exists(progress_path):
        with open(progress_path) as f:
            return set(json.load(f))
    return set()

def save_progress(done_batches):
    with open(progress_path, "w") as f:
        json.dump(sorted(done_batches), f)

def encode_batch_safely(docs, model, batch_size):
    """Encode with automatic batch-size backoff on CUDA OOM."""
    try:
        return model.encode(
            docs,
            batch_size=batch_size,
            prompt=EMBED_QUERY_PROMPT,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if batch_size <= 4:
            raise
        print(f"  OOM at batch_size={batch_size}, retrying at {batch_size // 2}")
        return encode_batch_safely(docs, model, batch_size // 2)

done_batches = load_progress()

# Open (or create) the combined memmap once. If it already exists from a previous
# run, open in "r+" so batches already written aren't wiped out.
if os.path.exists(COMBINED_EMBEDDINGS_PATH) and done_batches:
    all_embeddings = np.lib.format.open_memmap(COMBINED_EMBEDDINGS_PATH, mode="r+")
    assert all_embeddings.shape == (n_docs, embed_dim), (
        f"Existing embeddings file shape {all_embeddings.shape} doesn't match "
        f"expected {(n_docs, embed_dim)} — delete {COMBINED_EMBEDDINGS_PATH} and progress.json to start fresh."
    )
else:
    all_embeddings = np.lib.format.open_memmap(
        COMBINED_EMBEDDINGS_PATH, mode="w+", dtype=np.float32, shape=(n_docs, embed_dim)
    )

print(f"Encoding {n_docs:,} conversations with {EMBED_MODEL_NAME} "
      f"in {n_batches} batches of up to {EMBED_BATCH_SIZE:,} conversations each")
print(f"Resuming from checkpoint: {len(done_batches)}/{n_batches} batches already done")

for b in range(n_batches):
    if b in done_batches:
        print(f"[batch {b+1}/{n_batches}] already checkpointed, skipping")
        continue

    start = b * EMBED_BATCH_SIZE
    end = min(start + EMBED_BATCH_SIZE, n_docs)

    print(f"[batch {b+1}/{n_batches}] encoding conversations {start:,}-{end:,} ...")
    batch_embeddings = encode_batch_safely(all_docs[start:end], embed_model, encode_batch_size)

    # Write straight into the combined memmap and flush - this *is* the checkpoint,
    # so a crash right after this line loses nothing for batch b.
    all_embeddings[start:end] = batch_embeddings
    all_embeddings.flush()

    done_batches.add(b)
    save_progress(done_batches)
    print(f"[batch {b+1}/{n_batches}] checkpointed -> {COMBINED_EMBEDDINGS_PATH} (rows {start:,}-{end:,})")

    del batch_embeddings
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

print("\nAll batches encoded and checkpointed.")
print("Combined embeddings saved to:", COMBINED_EMBEDDINGS_PATH)
print("Embeddings shape:", all_embeddings.shape)


Longest docs (characters): [8780109, 8543799, 8428026, 5990999, 5179026, 4483158, 4426650, 3193197, 2680519, 2580374]
95th percentile length: 5,557 chars
Median length: 553 chars
Encoding 2,000,000 conversations with BAAI/bge-base-en-v1.5 in 20 batches of up to 100,000 conversations each
Resuming from checkpoint: 20/20 batches already done
[batch 1/20] already checkpointed, skipping
[batch 2/20] already checkpointed, skipping
[batch 3/20] already checkpointed, skipping
[batch 4/20] already checkpointed, skipping
[batch 5/20] already checkpointed, skipping
[batch 6/20] already checkpointed, skipping
[batch 7/20] already checkpointed, skipping
[batch 8/20] already checkpointed, skipping
[batch 9/20] already checkpointed, skipping
[batch 10/20] already checkpointed, skipping
[batch 11/20] already checkpointed, skipping
[batch 12/20] already checkpointed, skipping
[batch 13/20] already checkpointed, skipping
[batch 14/20] already checkpointed, skipping
[batch 15/20] already checkpointed, s

## 8. Fit BERTopic once on the full corpus

A single `fit_transform` call over all ~2.5M conversations from all 3 chunks combined — no
chunk-by-chunk fit/transform split, so topics that only show up strongly in chunk 1 or 2 still
get to form their own cluster instead of being forced into whatever chunk 0 happened to find.

## 9. Save results and export

`topic_info["Name"]` carries the LLM-written core-idea label (from the `"Main"`
representation model in step 5). `topic_info["KeyBERT"]` still has the raw keyword view if
you want to compare the two side by side.

In [ ]:
# ================================================================
# Configure RMM to use the full 80GB pool before any cuML calls
# ================================================================
import rmm

rmm.reinitialize(
    pool_allocator=True,
    initial_pool_size=int(70 * 1e9),   # 70GB pool, leaves headroom for PyTorch/embeddings/overhead
    maximum_pool_size=int(76 * 1e9),   # hard ceiling, leaves ~4GB for driver/other processes
)
print("RMM pool reinitialized for 80GB GPU")

# Free anything the embedding step was still holding
if 'embed_model' in dir():
    del embed_model
gc.collect()
torch.cuda.empty_cache()

RMM pool reinitialized for 80GB GPU


In [ ]:
umap_model, hdbscan_model = build_cluster_models()
if 'embed_model' not in dir() or embed_model is None:
    embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
    embed_model.max_seq_length = MAX_SEQ_LENGTH
topic_model = BERTopic(
    embedding_model=embed_model,          # embeddings already computed — skip re-embedding
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    representation_model=representation_model,
    calculate_probabilities=False,
    verbose=True,
)

print(f"Fitting BERTopic on all {len(all_docs):,} conversations ...")
all_topics, _ = topic_model.fit_transform(all_docs, all_embeddings)

topic_model.save(MODEL_PATH, serialization="safetensors", save_ctfidf=True, save_embedding_model=True)
print("Saved fitted model to:", MODEL_PATH)

# Save per-conversation topic assignment immediately — don't lose this if anything downstream fails
labels_df = pd.DataFrame({"conversation_id": all_ids, "topic": all_topics})
labels_path = os.path.join(OUTPUT_DIR, "conversation_topics.parquet")
labels_df.to_parquet(labels_path, index=False)
print("Saved topic labels to:", labels_path)
print(labels_df["topic"].value_counts().head(20))

del all_embeddings
gc.collect()
torch.cuda.empty_cache()

Using cuML UMAP + HDBSCAN (GPU)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fitting BERTopic on all 2,000,000 conversations ...


2026-07-11 11:48:55,954 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-11 12:04:50,679 - BERTopic - Dimensionality - Completed ✓
2026-07-11 12:04:50,744 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-11 12:06:21,975 - BERTopic - Cluster - Completed ✓
2026-07-11 12:06:22,468 - BERTopic - Representation - Fine-tuning topics using representation models.
  0%|          | 0/1296 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False 

Saved fitted model to: /content/drive/MyDrive/bertopic_outputs/bertopic_model
Saved topic labels to: /content/drive/MyDrive/bertopic_outputs/conversation_topics.parquet
topic
-1     1300557
 0       60209
 1       24857
 2       20546
 3       18303
 4       16912
 5       14699
 6       14303
 7       14240
 8       12953
 9       12891
 10      12558
 11      10922
 12       9087
 13       8837
 14       8786
 15       8715
 16       7406
 17       7138
 18       6594
Name: count, dtype: int64


In [ ]:
new_topics = topic_model.reduce_outliers(
    all_docs, all_topics,
    strategy="c-tf-idf",   # reassigns based on topic-word similarity
    threshold=0.1,
)
topic_model.update_topics(all_docs, topics=new_topics)

2026-07-11 15:04:10,732 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [ ]:
results_df = pd.DataFrame({"conversation_id": all_ids, "topic": all_topics})
results_path = os.path.join(OUTPUT_DIR, "conversation_topics.csv")
results_df.to_csv(results_path, index=False)
print("Saved combined results to:", results_path)

topic_info = topic_model.get_topic_info()
topic_info_path = os.path.join(OUTPUT_DIR, "topic_info.csv")
topic_info.to_csv(topic_info_path, index=False)
print("Saved topic info to:", topic_info_path)

print(topic_info[["Topic", "Count", "Name"]].head(20))


Saved combined results to: /content/drive/MyDrive/bertopic_outputs/conversation_topics.csv
Saved topic info to: /content/drive/MyDrive/bertopic_outputs/topic_info.csv
    Topic   Count                                               Name
0      -1  805663                        -1_https_co_conservative_in
1       0   62413        0_mehdirhasan_israel_netanyahu_vividprowess
2       1   26125             1_assassination_attempt_foxnews_secret
3       2   22294  2_conservative_bc_pierrepoilievre_cpac_tv_just...
4       3   20319                   3_elonmusk_iamnot_elon_elon_musk
5       4   17345           4_roshan_rinaldi_timhannan_covid_vaccine
6       5   14998                      5_iamsophianelson_vp_https_co
7       6   14608                          6_foxnews_https_co_kamala
8       7   14712               7_anniefortruth_e_galv_kamala_harris
9       8   15372           8_labour_reform_conservative_bladeofthes
10      9   15132           9_libsoftiktok_davidsacks_ukraine_russia
11   